# Rarity-driven sparse retrieval — diagnostics

| Section | What runs | Gate |
|---|---|---|
| Step 1 | `rarity` — IDF over the pool, irregular-term list | terms are scriptural vocabulary, not tokenizer debris |
| Step 3 | `leakage` — near-duplicate audit, quarantine | flag count small enough that quarantining leaves the pool intact |
| Step 2 | `sparse_select` — greedy coverage dry run | the channel fires often enough, and is less redundant than dense |


In [1]:
# e5-large in fp32 over a 10.8k-row pool; any Colab GPU is enough.
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA GeForce RTX 4060 Laptop GPU, 8188 MiB


In [2]:
%cd /home/prnamhr/projects/Style-Aware-MT
!pip install -r requirements.txt

# Text-only pipeline; these two carry an ABI mismatch against the pinned torch.
!pip uninstall -y torchvision torchaudio

/home/prnamhr/projects/Style-Aware-MT

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


`data/knn_index/` is git-ignored, so the pool index is rebuilt each session. The register
centroid is committed and already present.

In [3]:
!python3 manage.py build_index --config configs/base_qwen.yaml

Embedding 10860 Persian/Arabic training sources with intfloat/multilingual-e5-large-instruct ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Batches: 100%|███████████████████████████████| 340/340 [00:13<00:00, 25.51it/s]
Wrote index to data/knn_index/ : embeddings (10860, 1024), 10860 pairs


---
## Step 1 — the rarity list

IDF over the source side of `data/splits/train.jsonl`, keeping the rarest terms.

In [4]:
!python3 manage.py rarity --config configs/sparse_retrieval.yaml

Computing IDF over 10860 pool sources (zwnj=keep) ...
  22789 pool terms -> 10061 in df band [2, 20]
  44.1% of the vocabulary, observed df [2, 20]
  df histogram (irregular): {'1': 0, '2': 3669, '3': 1809, '4-9': 3357, '10-99': 1226, '100+': 0}
  ZWNJ variant collisions: 25 [('آسود\u200cگی', 'آسودگی'), ('الذین\u200cهم', 'الذینهم'), ('انشآء\u200cالله', 'انشآءالله')]
Wrote results/rarity_train.json and results/rarity_train_sample50.tsv


In [5]:
import json

import pandas as pd

rarity = json.load(open('results/rarity_train.json'))
print(f"{rarity['n_terms']} pool terms -> {rarity['n_irregular']} irregular "
      f"(requested {rarity['config']['top_frac']:.0%}, realized {rarity['realized_frac']:.1%}, "
      f"df <= {rarity['cutoff_df']})")
print('pool df histogram     :', rarity['df_histogram']['pool'])
print('irregular df histogram:', rarity['df_histogram']['irregular'])

sample = pd.read_csv('results/rarity_train_sample50.tsv', sep='\t')
sample['example'] = sample['example'].str.slice(0, 60)
sample

KeyError: 'top_frac'

### Normalization check — ZWNJ

In [ ]:
collisions = json.load(open('results/rarity_train.json'))['zwnj_collisions']
print(f'{len(collisions)} ZWNJ variant collisions')
pd.DataFrame(collisions, columns=['split spelling', 'joined spelling']).head(25)

25 ZWNJ variant collisions


,split spelling,joined spelling
0,آسود‌گی,آسودگی
1,الذین‌هم,الذینهم
2,انشآء‌الله,انشآءالله
3,این‌قدر,اینقدر
4,بیچار‌گان,بیچارگان
5,بی‌خبر,بیخبر
6,بی‌مثال,بیمثال
7,جان‌فزا,جانفزا
8,خون‌ریزی,خونریزی
9,راست‌گو,راستگو


---
## Step 3 — leakage audit

In [ ]:
!python manage.py leakage \
    --config configs/sparse_retrieval.yaml \
    --split val \
    --write-quarantine

In [ ]:
leak = json.load(open('results/leakage_val.json'))
print(f"val: {leak['n_eval_rows_flagged']}/{leak['n_eval_rows']} eval rows flagged, "
      f"{leak['n_pool_rows_flagged']} pool rows implicated")
print('  max-cos histogram:', leak['max_cos_histogram'])

pd.DataFrame([
    {'cos': f['cos'], 'jac_src': f['jaccard_source'], 'jac_tgt': f['jaccard_target'],
     'eval': f['eval_source'][:50], 'pool': f['pool_source'][:50]}
    for f in leak['flags'][:10]
])

In [ ]:
!python3 manage.py build_index --config configs/base_qwen.yaml \
    --index_dir data/knn_index_clean \
    --quarantine data/splits/pool_quarantine.json

---
## Step 2 — the sparse channel

In [ ]:
!python3 manage.py sparse_select --config configs/sparse_retrieval.yaml \
    --split val --index_dir data/knn_index_clean

In [18]:
sel = json.load(open('results/sparse_selection_val.json'))
print('routes            :', sel['route_fractions'])
print('irregular terms/query — mean', sel['query_terms']['mean'],
      'deciles', sel['query_terms']['deciles'])
print('share at or above :', sel['query_terms']['share_at_or_above'])
print('coverage (routed) :', sel['coverage']['mean'])
print('intra-set cosine  :', sel['intra_set_similarity'])

routes            : {'sparse': 0.0, 'hybrid': 0.0197, 'dense': 0.9803}
irregular terms/query — mean 0.777 deciles [0, 0, 0, 0, 0, 0, 1, 1, 1, 2, 7]
share at or above : {'1': 0.4981, '2': 0.1882, '3': 0.0582, '4': 0.0197, '5': 0.0091, '6': 0.003}
coverage (routed) : 1.0
intra-set cosine  : {'sparse': 0.8896, 'dense_baseline': 0.9002}


### Threshold sweep

`min_query_terms` is the one knob that decides whether the channel exists at all. Selection
is cheap once the index is loaded, so sweep it rather than arguing about it. Each run
writes its own report, leaving the configured run's above intact.

In [ ]:
for thr in (1, 2, 3, 4):
    print(f'--- min_query_terms={thr}')
    !python3 manage.py sparse_select --config configs/sparse_retrieval.yaml --split val --index_dir data/knn_index_clean --min_query_terms {thr} --out results/sparse_sweep_val_t{thr}.json 2>&1 | grep -E 'routes|intra-set'

### Worked examples

The trace behind three routed queries: which irregular terms the query carried, how much
of that rarity the selected set covered, and which exemplars were chosen.

In [20]:
for ex in sel['examples']:
    print('QUERY :', ex['source'][:90])
    print('  route', ex['trace']['route'], '| terms', ex['trace']['query_terms'],
          '| coverage', ex['trace']['coverage'])
    for e in ex['exemplars']:
        print('   -', e[:90])
    print()

QUERY : لذا اذکر لک بعض ما اکرمنی الله عمّا تطیقه النّفوس و تحمله العقول لئلّا یرفع ضوضآء المبغضین
  route hybrid | terms ['المبغضین', 'المنافقین', 'اکرمنی', 'تحمله'] | coverage 1.0
   - فارحمنی بجودک ثمّ اکرمنی بسلطانک ثمّ قرّبنی بألطافک.
   - ولکن انّا لا نحبّ بأن نذکر ما لا ذکر فی البیان لئلّا یرفع ضجیج المبغضین.
   - ان اثبتنی علی حبّک و رضائک علی شأن لا یمنعنی اعراض المشرکین من بریّتک و ضوضآء المنافقین من
   - لو ارید ان اذکر لک ما ورد علیّ لن تحمله النّفوس و لا العقول
   - یا حزب الله جهد نمائید شاید قلوب احزاب مختلفهٔ عالم بآب بردباری و شفقت شما از ضغینه و بغضا
   - کذلک نطق لسانی لأحد اغصانی و ذکرناه لأحبّائی الّذین نبذوا الأوهام و اخذوا ما امروا به فی ی
   - و امّا من چنین میگویم دشمنانتان را دوست دارید و ذکر خیر کنید بدگویان خود را و مبغضانتان را
   - ویلٌ لک یا ایُّها المشرک باللّه و للّذین اتّخذوک إماما لأنفسهم من دون بیّنة و لا کتاب مشهو

QUERY : و ان یقولون هذه الأسفار الّتی تکون بین یدی هذه الفئة و یسمّونها بالانجیل و ینسبونها بعیسی 
  route hybrid | terms ['الفئة', 'الف